In [5]:
%run bootstrap.ipynb

/Users/austinburt/miniforge3/envs/tf-metal/lib/python3.9/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


# THE VACUUM WORLD   

In this notebook, we will be discussing **the structure of agents** through an example of the **vacuum agent**. The job of AI is to design an **agent program** that implements the agent function: the mapping from percepts to actions. We assume this program will run on some sort of computing device with physical sensors and actuators: we call this the **architecture**:

<h3 align="center">agent = architecture + program</h3>

Before moving on, please review [<b>agents.ipynb</b>](https://github.com/aimacode/aima-python/blob/master/agents.ipynb)

## Persistent Memory Architecture

In [6]:
from dataclasses import dataclass, field

from aima.agents import *


@dataclass
class VacuumWorldInitialState:
    rows: int = 4
    cols: int = 4
    start: tuple[int, int] = (0, 0)
    obstacles: set[tuple[int, int]] = field(default_factory=lambda: {(1, 1)})
    dirty_cells: set[tuple[int, int]] = field(
        default_factory=lambda: {(0, 1), (2, 2), (3, 1)}
    )
    rewards: dict[tuple[int, int], int] = field(
        default_factory=lambda: {(3, 1): -1, (3, 3): 10}
    )
    grid: dict[tuple[int, int], object] = field(
        default_factory=lambda: {
            (0, 0): "start",
            (0, 1): 0,
            (0, 2): 0,
            (0, 3): 0,
            (1, 0): 0,
            (1, 1): "X",
            (1, 2): 0,
            (1, 3): 0,
            (2, 0): 0,
            (2, 1): 0,
            (2, 2): 0,
            (2, 3): 0,
            (3, 0): 0,
            (3, 1): -1,
            (3, 2): 0,
            (3, 3): 10,
        }
    )


initial_state = VacuumWorldInitialState()
print("Vacuum world initial state loaded.")
print("start:", initial_state.start)
print("obstacles:", sorted(initial_state.obstacles))
print("dirty cells:", sorted(initial_state.dirty_cells))
print("rewards:", sorted(initial_state.rewards.items()))

Vacuum world initial state loaded.
start: (0, 0)
obstacles: [(1, 1)]
dirty cells: [(0, 1), (2, 2), (3, 1)]
rewards: [((3, 1), -1), ((3, 3), 10)]


## Run the Persistent-Memory Agent

The environment owns the full grid and dirt state. The agent receives only its current cell, dirt flag, and adjacent-cell percepts. Each action and resulting position is printed below; the agent also prints its persistent-memory trace.

In [7]:
class VacuumWorldEnvironment:
    """Small simulator that exposes only local percepts to the agent."""

    directions = {
        "UP": (-1, 0),
        "DOWN": (1, 0),
        "LEFT": (0, -1),
        "RIGHT": (0, 1),
    }

    def __init__(self, initial_state):
        self.initial_state = initial_state
        self.position = initial_state.start
        self.dirty_cells = set(initial_state.dirty_cells)

    def percept(self):
        """Return the current cell and its adjacent values, not the full grid."""
        neighbors = {}
        for direction, (row_delta, col_delta) in self.directions.items():
            neighbor = (
                self.position[0] + row_delta,
                self.position[1] + col_delta,
            )
            row, col = neighbor
            inside_grid = (
                0 <= row < self.initial_state.rows
                and 0 <= col < self.initial_state.cols
            )
            if not inside_grid or neighbor in self.initial_state.obstacles:
                neighbors[direction] = None
            else:
                neighbors[direction] = self.initial_state.grid[neighbor]

        return {
            "pos": self.position,
            "cell": self.initial_state.grid[self.position],
            "dirty": self.position in self.dirty_cells,
            "neighbors": neighbors,
        }

    def execute_action(self, action):
        """Apply a cleaning or movement action without leaving the grid."""
        if action == "SUCK":
            self.dirty_cells.discard(self.position)
            return
        if action == "NO-OP":
            return
        if action not in self.directions:
            raise ValueError(f"Unsupported action: {action}")

        row_delta, col_delta = self.directions[action]
        next_position = (
            self.position[0] + row_delta,
            self.position[1] + col_delta,
        )
        row, col = next_position
        inside_grid = (
            0 <= row < self.initial_state.rows
            and 0 <= col < self.initial_state.cols
        )
        if inside_grid and next_position not in self.initial_state.obstacles:
            self.position = next_position


from agent_strategy1 import PersistentMemoryAgent

world = VacuumWorldEnvironment(initial_state)
agent = PersistentMemoryAgent(world)

print("Persistent memory initialized for this agent:")
for memory_name, memory_value in agent.memory.items():
    print(f"  {memory_name}: {memory_value!r}")

Persistent memory initialized for this agent:
  visited: set()
  obstacles: set()
  rewards: {}
  dirty_known: set()
  clean_known: set()
  last_pos: None
  last_action: None
  step_count: 0
  history: []


The memory dictionary is retained by the same agent instance across all turns:

- `visited` and `obstacles` record where the agent has been and what it cannot enter.
- `rewards` records observed adjacent penalty/reward values.
- `dirty_known` and `clean_known` track cells that still need cleaning or are known clean.
- `last_pos`, `last_action`, and `step_count` preserve turn-to-turn state.
- `history` stores the step, position, previous action, and a note for each memory update.

In [8]:
for turn in range(30):
    percept = world.percept()
    agent.update_memory(percept)
    action = agent.program(percept)
    old_position = world.position
    world.execute_action(action)
    print(
        f"Move {turn + 1:02d}: {old_position} --{action}--> "
        f"{world.position}; remaining dirty cells: {len(world.dirty_cells)}"
    )

print("Final position:", world.position)
print("Remaining dirty cells:", sorted(world.dirty_cells))
print("Final persistent memory:")
for memory_name, memory_value in agent.memory.items():
    print(f"  {memory_name}: {memory_value!r}")

[step 0] pos=(0, 0) action=None → visited (0, 0)
[step 0] pos=(0, 0) action=None → observed clean cell
[step 1] pos=(0, 0) action=DOWN → chose DOWN (reason: unvisited neighbor (1, 0))
Move 01: (0, 0) --DOWN--> (1, 0); remaining dirty cells: 3
[step 1] pos=(1, 0) action=DOWN → visited (1, 0)
[step 1] pos=(1, 0) action=DOWN → sensed obstacle at (1, 1)
[step 1] pos=(1, 0) action=DOWN → observed clean cell
[step 2] pos=(1, 0) action=DOWN → chose DOWN (reason: unvisited neighbor (2, 0))
Move 02: (1, 0) --DOWN--> (2, 0); remaining dirty cells: 3
[step 2] pos=(2, 0) action=DOWN → visited (2, 0)
[step 2] pos=(2, 0) action=DOWN → observed clean cell
[step 3] pos=(2, 0) action=DOWN → chose DOWN (reason: unvisited neighbor (3, 0))
Move 03: (2, 0) --DOWN--> (3, 0); remaining dirty cells: 3
[step 3] pos=(3, 0) action=DOWN → visited (3, 0)
[step 3] pos=(3, 0) action=DOWN → recorded reward -1 at (3, 1)
[step 3] pos=(3, 0) action=DOWN → observed clean cell
[step 4] pos=(3, 0) action=RIGHT → chose RIGH